In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# KHỞI TẠO VÀ BỔ SUNG CÁC THƯ VIỆN MACHINE LEARNING CÒN THIẾU:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Thư viện dùng để đánh giá kết quả mô hình
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
import os
from pathlib import Path
import glob

# =====================================================================
# BƯỚC 1: ĐỌC VÀ LÀM SẠCH TÊN CỘT (SỬA LỖI KEYERROR)
# =====================================================================
# 1. Đọc dữ liệu (Đưa lệnh này lên đầu tiên)
filename = 'dulieuketoan_clean.csv'
# Try direct paths and common data folders, then do a recursive search
candidates = [filename, os.path.join('data', filename), os.path.join('..', 'data', filename)]
candidates += glob.glob(f'**/{filename}', recursive=True)
candidates += glob.glob(f'**/*{Path(filename).stem}*.csv', recursive=True)

found = next((Path(p) for p in candidates if Path(p).is_file()), None)

if not found:
    # helpful debug info if file is missing
    cwd_files = '\n'.join(sorted([str(p) for p in Path('.').iterdir()]))
    raise FileNotFoundError(
        f"Could not find '{filename}'. Searched candidates: {candidates}\n"
        f"Files in working directory:\n{cwd_files}"
    )

df = pd.read_csv(found)

# 2. XÓA BỎ KHOẢNG TRẮNG THỪA TRONG TÊN CỘT
df.columns = df.columns.str.strip()

# 3. Chia tính năng (X) và biến mục tiêu (y) - Ép kiểu thành số nguyên
X = df.drop('Bankrupt?', axis=1)
y = df['Bankrupt?'].astype(int)

# 4. Chia tập dữ liệu (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Chuẩn hóa dữ liệu 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =====================================================================
# BƯỚC 2: HUẤN LUYỆN CÁC MÔ HÌNH
# =====================================================================
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    print(f"Đã huấn luyện xong: {name}")

# =====================================================================
# BƯỚC 3: ĐÁNH GIÁ KẾT QUẢ
# =====================================================================
print("\n" + "="*50 + "\nBÁO CÁO ĐÁNH GIÁ MÔ HÌNH\n" + "="*50)

results = {}
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {"Accuracy": acc, "ROC-AUC": auc}
    
    print(f"\n[MÔ HÌNH: {name}]")
    print(classification_report(y_test, y_pred, target_names=['Bình thường', 'Phá sản']))

# In bảng so sánh
df_results = pd.DataFrame(results).T
print("\nBẢNG SO SÁNH KẾT QUẢ TỔNG HỢP:")
print(df_results)

Đã huấn luyện xong: Logistic Regression
Đã huấn luyện xong: Random Forest


c:\Users\Asus\anaconda3\envs\distress_prediction\lib\site-packages\xgboost\training.py:200: UserWarning: [22:04:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Đã huấn luyện xong: XGBoost

BÁO CÁO ĐÁNH GIÁ MÔ HÌNH

[MÔ HÌNH: Logistic Regression]
              precision    recall  f1-score   support

 Bình thường       0.98      0.99      0.99      1320
     Phá sản       0.63      0.27      0.38        44

    accuracy                           0.97      1364
   macro avg       0.80      0.63      0.68      1364
weighted avg       0.97      0.97      0.97      1364


[MÔ HÌNH: Random Forest]
              precision    recall  f1-score   support

 Bình thường       0.97      1.00      0.98      1320
     Phá sản       0.62      0.18      0.28        44

    accuracy                           0.97      1364
   macro avg       0.79      0.59      0.63      1364
weighted avg       0.96      0.97      0.96      1364


[MÔ HÌNH: XGBoost]
              precision    recall  f1-score   support

 Bình thường       0.98      0.99      0.98      1320
     Phá sản       0.58      0.25      0.35        44

    accuracy                           0.97      1